# YFinanceSQLite

In [ ]:
import yfinance as yf
import sqlite3
import pandas as pd
from datetime import datetime, timedelta
import logging
from typing import Literal

In [ ]:
def setup_database(conn):
    """Create tables if they don't exist."""
    cursor = conn.cursor()

    # Daily stock data table
    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS stock_data_daily (
            symbol TEXT NOT NULL,
            date DATE NOT NULL,
            open REAL,
            high REAL,
            low REAL,
            close REAL,
            volume INTEGER,
            PRIMARY KEY (symbol, date)
        )
    """
    )

    # Weekly stock data table
    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS stock_data_weekly (
            symbol TEXT NOT NULL,
            date DATE NOT NULL,
            open REAL,
            high REAL,
            low REAL,
            close REAL,
            volume INTEGER,
            PRIMARY KEY (symbol, date)
        )
    """
    )

    # Intraday stock data table (15min, 1min, etc.)
    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS stock_data_intraday (
            symbol TEXT NOT NULL,
            datetime TIMESTAMP NOT NULL,
            interval TEXT NOT NULL,
            open REAL,
            high REAL,
            low REAL,
            close REAL,
            volume INTEGER,
            PRIMARY KEY (symbol, datetime, interval)
        )
    """
    )

    # Metadata table to track last update times for different intervals
    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS update_log (
            symbol TEXT NOT NULL,
            interval TEXT NOT NULL,
            last_updated TIMESTAMP,
            last_datetime TIMESTAMP,
            PRIMARY KEY (symbol, interval)
        )
    """
    )

    # Create indexes for faster queries\
    cursor.execute(
        """
        CREATE INDEX IF NOT EXISTS idx_daily_symbol_date 
        ON stock_data_daily (symbol, date)
    """
    )

    cursor.execute(
        """
        CREATE INDEX IF NOT EXISTS idx_weekly_symbol_date 
        ON stock_data_weekly (symbol, date)
    """
    )

    cursor.execute(
        """
        CREATE INDEX IF NOT EXISTS idx_intraday_symbol_datetime 
        ON stock_data_intraday (symbol, datetime)
    """
    )

    cursor.execute(
        """
        CREATE INDEX IF NOT EXISTS idx_intraday_symbol_interval 
        ON stock_data_intraday (symbol, interval, datetime)
    """
    )

    conn.commit()

In [ ]:
def get_last_datetime(conn, symbol, interval):
    """Get the last datetime we have data for a symbol and interval."""
    cursor = conn.cursor()
    cursor.execute(
        """
        SELECT last_datetime FROM update_log 
        WHERE symbol = ? AND interval = ?
    """,
        (symbol, interval),
    )

    result = cursor.fetchone()
    return result[0] if result else None

In [ ]:
def _is_daily_interval(interval):
    """Check if interval is daily."""
    return interval == "1d"


def _is_weekly_interval(interval):
    """Check if interval is weekly."""
    return interval == "1wk"

In [ ]:
def _fetch_full(conn, symbol, interval, period, is_daily, is_weekly):
    """
    Wipe existing data for the symbol/interval and download a fresh full history.

    The wipe is committed before the download so a mid-run crash never leaves
    a stale last_datetime in update_log that would cause the next incremental
    run to silently skip a re-fetch.
    """
    print(f"  Force full update ({period})")

    # Clear existing data for this symbol/interval
    if is_daily:
        conn.execute("DELETE FROM stock_data_daily WHERE symbol = ?", (symbol,))
    elif is_weekly:
        conn.execute("DELETE FROM stock_data_weekly WHERE symbol = ?", (symbol,))
    else:
        conn.execute(
            "DELETE FROM stock_data_intraday WHERE symbol = ? AND interval = ?",
            (symbol, interval),
        )
    
    # Clear update_log
    conn.execute(
        "DELETE FROM update_log WHERE symbol = ? AND interval = ?",
        (symbol, interval),
    )

    # Commit the wipe before we start downloading
    conn.commit()  

    return yf.Ticker(symbol).history(period=period, interval=interval)

In [ ]:
def _fetch_incremental(conn, symbol, interval, period, is_daily, is_weekly):
    """
    Download only data newer than what is already stored, or a full history
    if this is the first run for the symbol/interval.
    """
    last_datetime = get_last_datetime(conn, symbol, interval)

    # First time downloading this symbol/interval
    if not last_datetime:
        print(f"  Full download ({period})")
        return yf.Ticker(symbol).history(period=period, interval=interval)

    # Daily data
    if is_daily or is_weekly:
        lookback_days = 5 if is_daily else 21
        start_date = (
            pd.to_datetime(last_datetime).date() - timedelta(days=lookback_days)
        ).strftime("%Y-%m-%d")
        print(f"  Incremental update from {start_date}")
        return yf.Ticker(symbol).history(start=start_date, interval=interval)
    
    # Intraday: fetch a rolling window then filter to strictly new rows
    print(f"  Incremental update from {last_datetime}")
    data = yf.Ticker(symbol).history(period="5d", interval=interval)

    if not data.empty:
        # Convert last_datetime to timezone-aware for comparison
        last_dt = pd.to_datetime(last_datetime)
        if last_dt.tz is None and data.index.tz is not None:
            # If stored datetime is naive but data is timezone-aware, localize it
            last_dt = last_dt.tz_localize(data.index.tz)
        elif last_dt.tz is not None and data.index.tz is None:
            # If stored datetime is timezone-aware but data is naive, remove timezone
            last_dt = last_dt.tz_localize(None)

        # Filter to only new data
        data = data[data.index > last_dt]
    
    return data

In [ ]:
def _upsert(conn, table_name, df):
    """Insert rows, replacing on PRIMARY KEY conflict."""
    cols = list(df.columns)
    placeholders = ", ".join(["?"] * len(cols))
    col_list = ", ".join(cols)
    sql = (
        f"INSERT OR REPLACE INTO {table_name} ({col_list}) "
        f"VALUES ({placeholders})"
    )
    conn.executemany(sql, df.itertuples(index=False, name=None))

In [ ]:
def _update_log(conn, symbol, interval, last_point):
    """Record the latest datetime seen for this symbol/interval."""
    conn.execute(
        """
        INSERT OR REPLACE INTO update_log (symbol, interval, last_updated, last_datetime)
        VALUES (?, ?, ?, ?)
        """,
        (symbol, interval, datetime.now(), last_point),
    )

In [ ]:
def _prepare_and_store(conn, table_name, data, symbol, interval, is_eod):
    """
    Normalise columns, attach metadata, call upsert, and return the latest
    timestamp so the caller can update update_log.
    """
    data = data.copy()
    data.reset_index(inplace=True)
    data["symbol"] = symbol

    if is_eod:
        # Daily data processing
        data["Date"] = data["Date"].dt.date  # Convert to date only
        data = data.rename(
            columns={
                "Date": "date",
                "Open": "open",
                "High": "high",
                "Low": "low",
                "Close": "close",
                "Volume": "volume",
            }
        )
        columns = ["symbol", "date", "open", "high", "low", "close", "volume"]
        data = data[columns]
        last_point = data["date"].max()
    else:
        # Intraday data processing
        data["interval"] = interval
        data = data.rename(
            columns={
                "Datetime": "datetime",
                "Open": "open",
                "High": "high",
                "Low": "low",
                "Close": "close",
                "Volume": "volume",
            }
        )
        # Convert datetime to UTC then to string for consistent storage
        data["datetime"] = data["datetime"].dt.tz_convert("UTC").dt.strftime("%Y-%m-%dT%H:%M:%S%z")

        columns = [
            "symbol",
            "datetime",
            "interval",
            "open",
            "high",
            "low",
            "close",
            "volume",
        ]
        data = data[columns]
        last_point = data["datetime"].max()

    _upsert(conn, table_name, data)
    return last_point

In [ ]:
def update_symbol_data(
    conn, symbol, interval="1d", period="1y", force_full_update=False
):
    """
    Download and store data for a symbol with incremental updates.

    Args:
        symbol: Stock symbol (e.g., 'AAPL')
        interval: Data interval ('1m', '2m', '5m', '15m', '30m', '60m',
                          '90m', '1h', '1d', '5d', '1wk', '1mo', '3mo')
        period: Period to download if no existing data
        force_full_update: If True, re-download all data
    """
    print(f"Processing {symbol} ({interval})...")

    try:
        # Determine which table to use
        is_daily = _is_daily_interval(interval)
        is_weekly = _is_weekly_interval(interval)
        is_eod    = is_daily or is_weekly  # date-only tables

        if is_daily:
            table_name = "stock_data_daily"
        elif is_weekly:
            table_name = "stock_data_weekly"
        else:
            table_name = "stock_data_intraday" 

        if force_full_update:
            data = _fetch_full(conn, symbol, interval, period, is_daily, is_weekly)
        else:
            data = _fetch_incremental(conn, symbol, interval, period, is_eod, is_daily)     

        if data is None or data.empty:
            print(f"  No new data available for {symbol} ({interval})")
            return False

        last_point = _prepare_and_store(conn, table_name, data, symbol, interval, is_eod)
        _update_log(conn, symbol, interval, last_point)

        conn.commit()
        print(f"  Successfully stored {len(data)} records for {symbol} ({interval})")
        return True

    except Exception as e:
        print(f"  Error updating {symbol} ({interval}): {str(e)}")
        conn.rollback()
        return False

In [ ]:
def update_multiple_symbols(
    conn, symbols, interval="1d", period="1y", force_full_update=False
):
    """Update data for multiple symbols."""
    successful = []
    failed = []

    for symbol in symbols:
        if update_symbol_data(conn, symbol, interval, period, force_full_update):
            successful.append(symbol)
        else:
            failed.append(symbol)

    print(f"\nSummary for {interval}:")
    print(f"  Successful: {len(successful)} symbols")
    print(f"  Failed: {len(failed)} symbols")
    if failed:
        print(f"  Failed symbols: {', '.join(failed)}")

In [ ]:
def get_data(conn, symbol, interval="1d", start_date=None, end_date=None):
    """Retrieve data for a symbol from the database."""
    is_daily = _is_daily_interval(interval)
    is_weekly = _is_weekly_interval(interval)

    if is_daily:
        table = "stock_data_daily"
        date_col = "date"
        query = f"SELECT * FROM {table} WHERE symbol = ?"
        params = [symbol]
    elif is_weekly:
        table = "stock_data_weekly"
        date_col = "date"
        query = f"SELECT * FROM {table} WHERE symbol = ?"
        params = [symbol]
    else:
        table = "stock_data_intraday"
        date_col = "datetime"
        query = f"SELECT * FROM {table} WHERE symbol = ? AND interval = ?"
        params = [symbol, interval]

    if start_date:
        query += f" AND {date_col} >= ?"
        params.append(start_date)

    if end_date:
        query += f" AND {date_col} <= ?"
        params.append(end_date)

    query += f" ORDER BY {date_col}"

    df = pd.read_sql_query(query, conn, params=params)
    if not df.empty:
        # Parse datetime with explicit UTC handling
        if is_daily or is_weekly:
            # For daily data (no timezone)
            df[date_col] = pd.to_datetime(df[date_col], utc=False)
        else:
            # For intraday data stored as UTC strings
            df[date_col] = pd.to_datetime(df[date_col], utc=True)
        df.set_index(date_col, inplace=True)

    return df

In [ ]:
def get_available_symbols(conn, interval="1d"):
    """Get list of all symbols in the database for a specific interval."""
    is_daily = _is_daily_interval(interval)
    is_weekly = _is_weekly_interval(interval)

    if is_daily:
        query = "SELECT DISTINCT symbol FROM stock_data_daily ORDER BY symbol"
    elif is_weekly:
        query = "SELECT DISTINCT symbol FROM stock_data_weekly ORDER BY symbol"
    else:
        query = "SELECT DISTINCT symbol FROM stock_data_intraday WHERE interval = ? ORDER BY symbol"

    cursor = conn.cursor()
    if is_daily:
        cursor.execute(query)
    elif is_weekly:
        cursor.execute(query)
    else:
        cursor.execute(query, (interval,))

    return [row[0] for row in cursor.fetchall()]

In [ ]:
def get_data_info(conn):
    """Get summary information about stored data."""
    cursor = conn.cursor()

    # Daily data info
    cursor.execute(
        """
        SELECT 
            d.symbol,
            'daily' AS interval,
            COUNT(*) AS record_count,
            MIN(date) AS first_date,
            MAX(date) AS last_date,
            l.last_updated
        FROM stock_data_daily AS d 
        LEFT JOIN update_log AS l ON d.symbol = l.symbol AND l.interval = '1d'
        GROUP BY d.symbol 
    """
    )
    daily_data = cursor.fetchall()

    # Weekly data info
    cursor.execute("""
        SELECT w.symbol,
              'weekly' AS interval,
               COUNT(*) AS record_count,
               MIN(date) AS first_date,
               MAX(date) AS last_date,
               l.last_updated
        FROM stock_data_weekly AS w
        LEFT JOIN update_log AS l ON w.symbol = l.symbol AND l.interval = '1wk'
        GROUP BY w.symbol
    """)
    weekly_data = cursor.fetchall()

    # Intraday data info
    cursor.execute(
        """
        SELECT 
            symbol,
            interval,
            COUNT(*) as record_count,
            MIN(datetime) as first_date,
            MAX(datetime) as last_date,
            last_updated
        FROM stock_data_intraday 
        LEFT JOIN update_log USING (symbol, interval)
        GROUP BY symbol, interval 
    """
    )
    intraday_data = cursor.fetchall()

    # Combine results
    all_data = daily_data + weekly_data + intraday_data
    columns = [
        "Symbol",
        "Interval",
        "Records",
        "First Date",
        "Last Date",
        "Last Updated",
    ]

    return pd.DataFrame(all_data, columns=columns).sort_values(["Symbol", "Interval"])

In [ ]:
def cleanup_old_intraday_data(conn, days_to_keep=30):
    """Remove intraday data older than specified days."""
    cutoff_date = datetime.now() - timedelta(days=days_to_keep)
    cursor = conn.cursor()

    cursor.execute(
        """
        DELETE FROM stock_data_intraday 
        WHERE datetime < ?
    """,
        (cutoff_date,),
    )

    deleted_count = cursor.rowcount
    conn.commit()

    print(f"Deleted {deleted_count} intraday records older than {days_to_keep} days")
    return deleted_count

In [ ]:
def close(conn):
    """Close the database connection."""
    conn.close()

## Examples

In [ ]:
db_path = "stocks.db"
conn = sqlite3.connect(db_path)
setup_database(conn)

In [ ]:
# ── Daily data example ──────────────────────────────────────────
#  Define symbols to track
# symbols_1d = ["^SPX", "^VIX9D", "^VIX", "^VIX3M", "^VIX6M", "^VIX1Y", "ES=F"]
symbols_1d = ["^SPX"]

# Update daily data
print("Updating daily data...")
update_multiple_symbols(conn, symbols_1d, interval="1d", period="1y")

# Example: Get daily data
print(f"\nSample daily data for SPX:")
daily_data = get_data(conn, "^SPX", interval="1d")
print(daily_data.tail())

In [ ]:
# ── Weekly data example ──────────────────────────────────────────
symbols_1w = ["^SPX"]
full_update = True

# Update daily data
print("Updating weekly data...")
update_multiple_symbols(conn, symbols_1w, interval="1wk", period="5y", force_full_update=full_update)

# Example: Get daily data
print(f"\nSample weekly data for SPX:")
weekly_data = get_data(conn, "^SPX", interval="1wk")
print(weekly_data.tail())

In [ ]:
# ── Intraday data example ──────────────────────────────────────────
# symbols_15m = ["^SPX", "^VIX", "ES=F"]
symbols_15m = ["^SPX"]

# Update 15-minute data (note: limited historical data available)
print("\nUpdating 15-minute data...")
update_multiple_symbols(conn, symbols_15m, interval="15m", period="7d")

# Example: Get 15-minute data
print(f"\nSample 15-minute data for SPX:")
intraday_data = get_data(conn, "^SPX", interval="15m")
print(intraday_data.tail())

In [ ]:
# Show summary of stored data
print("\nData Summary:")
print(get_data_info(conn))

In [ ]:
# Clean up old intraday data (optional)
cleanup_old_intraday_data(conn, days_to_keep=7)

In [ ]:
# Close the database connection
close(conn)